# 04 - Validacao do Pipeline de Treino e Avaliacao

Este notebook valida as tarefas 11 e 12:

1. Pipeline de treino.
2. Pipeline de avaliacao.
3. Salvamento de checkpoint.
4. Salvamento de historico e metricas.

Ele usa os DataLoaders reais quando os splits existem. Para treino completo dos modelos do PBL, os proximos notebooks usarao esse mesmo pipeline.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
from torch import nn

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.datasets import DataLoaderConfig, create_dataloaders
from src.training.evaluate import evaluate_model
from src.training.train import TrainConfig, train_model

config.ensure_project_directories()
config.seed_everything()

print("DEVICE:", config.DEVICE)

## Modelo pequeno para validacao

Este modelo nao e a CNN final do PBL. Ele serve apenas para validar que o pipeline executa, salva checkpoint e calcula metricas.

In [ ]:
class TinyValidationCNN(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 8, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(16, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x).squeeze(1)

## Treino curto

Execute esta celula apenas depois de gerar `data/splits/train.csv`, `val.csv` e `test.csv`.

In [ ]:
required_splits = [config.SPLITS_DIR / f"{split}.csv" for split in ["train", "val", "test"]]
missing_splits = [path for path in required_splits if not path.exists()]

if missing_splits:
    raise FileNotFoundError(
        "Splits ausentes: "
        + ", ".join(str(path) for path in missing_splits)
        + ". Execute notebooks/02_preprocessamento_splits.ipynb primeiro."
    )

loader_config = DataLoaderConfig(batch_size=8, num_workers=0, pin_memory=torch.cuda.is_available())
loaders = create_dataloaders(dataloader_config=loader_config)

model = TinyValidationCNN()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_config = TrainConfig(
    model_name="tiny_validation_cnn",
    epochs=1,
    device=config.DEVICE,
    use_amp=torch.cuda.is_available(),
)

train_result = train_model(
    model=model,
    train_loader=loaders["train"],
    val_loader=loaders["val"],
    criterion=criterion,
    optimizer=optimizer,
    train_config=train_config,
)

train_result["summary"]

## Avaliacao no teste

A avaliacao calcula accuracy, precision, recall, F1, AUC, matriz de confusao, curva ROC e tempo medio de inferencia.

In [ ]:
test_metrics = evaluate_model(
    model=model,
    dataloader=loaders["test"],
    model_name="tiny_validation_cnn",
    device=config.DEVICE,
    output_dir=config.METRICS_DIR,
    checkpoint_path=train_result["best_checkpoint_path"],
)

test_metrics

## Proxima Etapa

Depois de validar este pipeline, seguir para as tarefas 13 e 14:

1. Implementar e treinar CNN propria.
2. Implementar e treinar CNN padrao.